# Daniyal Khan || 221A061 || 19

In [ ]:
# Daniyal Khan || 221A061

!pip install yt-dlp youtube-comment-downloader pandas plotly textblob
import pandas as pd
import plotly.express as px
from textblob import TextBlob
from youtube_comment_downloader import YoutubeCommentDownloader
import yt_dlp


In [2]:
# Daniyal Khan || 221A061

video_url = "https://youtu.be/oCMsejd6ovM?si=8Sr3D1dPsElPPzMG"

In [3]:
# Daniyal Khan || 221A061

ydl_opts = {}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
   info = ydl.extract_info(video_url, download=False)

print("Title:", info['title'])
print("Views:", info['view_count'])
print("Likes:", info.get('like_count', 'N/A'))


[youtube] Extracting URL: https://youtu.be/oCMsejd6ovM?si=8Sr3D1dPsElPPzMG
[youtube] oCMsejd6ovM: Downloading webpage


[youtube] oCMsejd6ovM: Downloading android vr player API JSON
Title: React JS 19 Tutorial in Hindi #38 useRef Hook
Views: 26254
Likes: 569


In [4]:
# Daniyal Khan || 221A061

downloader = YoutubeCommentDownloader()

comments = []
for comment in downloader.get_comments_from_url(video_url):
   comments.append(comment['text'])
   if len(comments) > 200:   # limit comments
       break

df = pd.DataFrame(comments, columns=['Comment'])
df.head()


,Comment
0,Please support me by like share comment and su...
1,grt
2,Excellent and simple explanation bro\nThanks a...
3,Please explain in more detail when we should u...
4,"sirrr ,,,your teaching style just"


In [5]:
# Daniyal Khan || 221A061

def get_sentiment(text):
   return TextBlob(text).sentiment.polarity

df['Sentiment'] = df['Comment'].apply(get_sentiment)

def label(score):
   if score > 0:
       return "Positive"
   elif score < 0:
       return "Negative"
   else:
       return "Neutral"

df['Label'] = df['Sentiment'].apply(label)


In [6]:
# Daniyal Khan || 221A061

# 1. Sentiment Pie Chart
fig = px.pie(df, names='Label', title='YouTube Comment Sentiment')
fig.show()

# 2. Sentiment Distribution
fig = px.histogram(df, x='Sentiment', title='Sentiment Score Distribution')
fig.show()

# 3. Top Words Used
from collections import Counter
import re

words = []
for comment in df['Comment']:
   words += re.findall(r'\w+', comment.lower())

common_words = Counter(words).most_common(10)

df_words = pd.DataFrame(common_words, columns=['Word', 'Count'])

fig = px.bar(df_words, x='Word', y='Count', title='Top Words in Comments')
fig.show()


In [7]:
# Daniyal Khan || 221A061

summary = df['Label'].value_counts()

print("📊 REPORT SUMMARY")
print(summary)

print("\nTotal Comments:", len(df))
print("Positive %:", round(summary.get('Positive',0)/len(df)*100,2))
print("Negative %:", round(summary.get('Negative',0)/len(df)*100,2))
print("Neutral %:", round(summary.get('Neutral',0)/len(df)*100,2))


📊 REPORT SUMMARY
Label
Positive    16
Neutral      7
Name: count, dtype: int64

Total Comments: 23
Positive %: 69.57
Negative %: 0.0
Neutral %: 30.43


In [8]:
# Daniyal Khan || 221A061

df.to_csv("youtube_analysis_report.csv", index=False)

from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
from collections import Counter
import re


# Sentiment count
sentiment_counts = df['Label'].value_counts()

# Top words
words = []
for comment in df['Comment']:
    words += re.findall(r'\w+', comment.lower())

common_words = Counter(words).most_common(10)
words_df = pd.DataFrame(common_words, columns=['Word', 'Count'])

# ---- Create Dashboard ----

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Sentiment Distribution",
        "Sentiment Score",
        "Top Words",
        "Sample Sentiment Scatter"
    ),
    specs=[[{"type": "domain"}, {"type": "xy"}],
           [{"type": "xy"}, {"type": "xy"}]]
)

# 1️⃣ Pie Chart
fig.add_trace(
    go.Pie(labels=sentiment_counts.index, values=sentiment_counts.values),
    row=1, col=1
)

# 2️⃣ Histogram
fig.add_trace(
    go.Histogram(x=df['Sentiment']),
    row=1, col=2
)

# 3️⃣ Bar Chart (Top Words)
fig.add_trace(
    go.Bar(x=words_df['Word'], y=words_df['Count']),
    row=2, col=1
)

# 4️⃣ Scatter Plot
fig.add_trace(
    go.Scatter(y=df['Sentiment'], mode='markers'),
    row=2, col=2
)

# Layout
fig.update_layout(
    height=800,
    width=1000,
    title_text="📊 YouTube Social Media Dashboard",
    showlegend=False
)

fig.show()
